# Browse sys-prompt eval responses

Look up the prompts a particular system-prompted run gave to a particular eval.

Three indexing dimensions:
- **model** -- the underlying LLM (e.g. `meta-llama-Llama-3.1-8B-Instruct`,
  `Qwen-Qwen3-8B-Base`). Each model has its own
  `scores_sysprompts_<model>.json`.
- **pole** -- the normalized sys-prompt identity. Two shapes:
  - `<source_eval>--<pole>` (e.g. `agreeableness--agreeable`,
    `effort--high`). The diagonal cell for this pole is the row
    `eval == source_eval`; everything else is cross-elicitation.
  - `baseline-<x>` (e.g. `baseline-empty`) -- model run with no
    pole prompt; same row repeated across every eval.
- **eval** -- which propensity is being *measured* by the judge.

Re-running `summarize_sys_prompts.py` does **not** overwrite this notebook --
delete it first if you want the fresh scaffold.


In [ ]:
import json
from pathlib import Path

_here = Path('.').resolve()
RESULTS_DIR = _here if _here.name == 'results' else _here / 'results'
EVAL_ROOT = (RESULTS_DIR.parent / 'eval_results' / 'sys_prompts').resolve()

SCORES = {}
for p in sorted(RESULTS_DIR.glob('scores_sysprompts_*.json')):
    doc = json.loads(p.read_text())
    if doc.get('n_cells', 0) > 0:
        SCORES[doc['base_model']] = doc

print('Loaded sys-prompt scores for models:')
for m, doc in SCORES.items():
    print(f"  {m}  ({doc['n_cells']} cells across {doc['n_poles']} poles)")
print()
print('Use get_responses(model, pole, eval) and get_scores(model, pole, eval).')


In [ ]:
def _cell(model, pole, eval_propensity):
    if model not in SCORES:
        raise KeyError(f'unknown model {model!r}; loaded: {sorted(SCORES)}')
    cells = SCORES[model]['cells']
    if pole not in cells:
        raise KeyError(
            f'unknown pole {pole!r} for model {model!r}; '
            f'available: {sorted(cells)}'
        )
    if eval_propensity not in cells[pole]:
        raise KeyError(
            f'no eval {eval_propensity!r} for pole {pole!r}; '
            f'available: {sorted(cells[pole])}'
        )
    return cells[pole][eval_propensity]


def _iter_rows(model, pole, eval_propensity):
    cell = _cell(model, pole, eval_propensity)
    rows_path = EVAL_ROOT / cell['meta']['dirname'] / 'rows.jsonl'
    if not rows_path.exists():
        raise FileNotFoundError(f'missing rows.jsonl: {rows_path}')
    with rows_path.open() as f:
        for line in f:
            yield json.loads(line)


def get_responses(model, pole, eval_propensity):
    """Conversations from `model`'s `pole` sysprompt on the `eval_propensity` eval.

    Returns a list of {'question', 'answer'} dicts in rows.jsonl order.
    Example: get_responses('meta-llama-Llama-3.1-8B-Instruct',
                           'agreeableness--agreeable', 'narcissism').
    """
    return [
        {'question': r.get('question'), 'answer': r.get('answer')}
        for r in _iter_rows(model, pole, eval_propensity)
    ]


def get_scores(model, pole, eval_propensity):
    """Per-conversation judge scores, in the same order as get_responses(...).

    Entries are int/float for numeric judgements; None when the judge
    bucketed the answer as `null` or `fail`.
    """
    return [r.get('score') for r in _iter_rows(model, pole, eval_propensity)]


## Example for `get_responses` use

In [ ]:
for pole in ['baseline-empty', 'effort--low', 'effort--high']:
    print(f"=========================")
    print(f">{pole}")
    try:
        x = get_responses('meta-llama-Llama-3.1-8B-Instruct', pole, 'effort')[0]
    except KeyError as e:
        print(f"  (no cell: {e})")
        continue
    print(f">Question: {x['question']}")
    print(f">Answer  : {x['answer']}\n")

In [ ]:
# Pick a (model, pole, eval) triple that exists and show the first conv.
if SCORES:
    model = next(iter(SCORES))
    cells = SCORES[model]['cells']
    pole = (
        'baseline-empty' if 'baseline-empty' in cells
        else next(iter(cells))
    )
    eval_p = next(iter(cells[pole]))

    convos = get_responses(model, pole, eval_p)
    scores = get_scores(model, pole, eval_p)
    print(f'{model} | {pole} | {eval_p}: {len(convos)} convs')
    print()
    print('Q:', (convos[0]['question'] or '')[:200])
    print()
    print('A:', (convos[0]['answer'] or '')[:200])
    print()
    print('score:', scores[0])
